In [1]:
# Standard library imports
import gc
import logging
import os
import sys
from pathlib import Path
import tempfile

# Third-party imports
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tqdm import tqdm
import zarr
from numcodecs import Blosc
import openpyxl
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from torch.cuda.amp import autocast
import torchvision.models as tvm
from sklearn.utils.class_weight import compute_class_weight

# Jupyter-specific imports
from IPython.display import display, HTML, Image

In [3]:
from src.dataloader.discover_wsi import discover_wsi_paths
from src.dataloader.discover_xml import discover_xml_paths
from src.dataloader.wsi_reader import WSIReader, read_wsi_region
from src.preprocessing.macenko import get_macenko_vectors, macenko_stain_normalize, estimate_reference_stain_vectors
from src.preprocessing.roi_utils import parse_asap_xml, patch_in_roi
from src.preprocessing.processing import normalize_images_to_zarr
from src.preprocessing.xml_to_mask import get_mask
from src.dataloader.zarr_patch_dataset import ZarrPatchDataset
from src.training.multitask_classifier import MultiTaskLesionClassifier
from src.training.train_phase1 import run_training
from src.training.train_phase2 import run_training_phase2
from src.training.train_unet import run_training_unet

In [4]:
# ============================================================================
# PATCH EXTRACTION CONFIGURATION
# ============================================================================

# Reference calculation parameters
PATCHES_PER_SLIDE = 20        # Random patches per slide for reference vectors
PATCH_SIZE = 256              # Patch size for reference calculation
LEVEL_REFERENCE = 2           # WSI pyramid level for reference (lower res, faster)
SEED = 42                     # Random seed for reproducibility

# Training patch parameters
PATCH_SIZE_TRAIN = 512        # High-res patch size for training
PATCH_STRIDE = 512            # Non-overlapping patches (stride = patch size)
PATCH_LIMIT_PER_SLIDE = None  # Optional: cap patches per slide (None = unlimited)
LEVEL_ANALYSIS = 0            # WSI pyramid level for training patches (full res)

# Backend and processing
USE_CUCIM = False             # Use CuCIM for faster WSI reading (False = OpenSlide)
PATCH_USE_GPU = True          # Use GPU for Macenko normalization if available

# ROI filtering
FILTER_BY_ROI = True          # Only extract patches within annotated ROIs
MASK_DOWNSAMPLE = 4           # Downsample factor for mask generation (memory/speed)

# ============================================================================
# DATASET SPLIT CONFIGURATION
# ============================================================================

# Patch normalization split
MAX_SLIDES_TO_PROCESS = None    # Number of slides to randomly sample
SAMPLE_SEED = 42              # Seed for slide sampling
SPLIT_NORMALIZED_RATIO = 0.20 # 20% normalized patches, 80% raw patches

# Train/Val/Test split for model training
TRAIN_RATIO = 0.70            # 70% training samples
VAL_RATIO = 0.10              # 10% validation samples
TEST_RATIO = 0.20             # 20% test samples
SPLIT_SEED = 42               # Seed for reproducible splits

# ============================================================================
# ZARR STORAGE CONFIGURATION
# ============================================================================

# Primary storage location
PATCH_ZARR_OUTPUT = "D:\\zarr"

# Zarr optimization settings
PATCH_CHUNK = (128, PATCH_SIZE_TRAIN, PATCH_SIZE_TRAIN, 3)  # Chunk size for I/O
WRITE_BUFFER_SIZE = 64        # Number of patches to buffer before writing

# Computed paths (normalized and raw outputs)
PATCH_ZARR_OUTPUT_NORM = Path(PATCH_ZARR_OUTPUT) / "patches_norm.zarr"
PATCH_ZARR_OUTPUT_RAW = Path(PATCH_ZARR_OUTPUT) / "patches_raw.zarr"

# Fallback storage (if primary drive is full)
PATCH_ZARR_FALLBACK = Path("D:/zarr_fallback")
#PATCH_ZARR_FALLBACK.mkdir(parents=True, exist_ok=True)
PATCH_ZARR_OUTPUT_NORM_FALLBACK = PATCH_ZARR_FALLBACK / "patches_norm.zarr"
PATCH_ZARR_OUTPUT_RAW_FALLBACK = PATCH_ZARR_FALLBACK / "patches_raw.zarr"

# ============================================================================
# OUTPUT DIRECTORIES
# ============================================================================

OUTPUT_BASE = Path('outputs/preprocessing')
OUTPUT_BASE.mkdir(parents=True, exist_ok=True)

# ============================================================================
# TRAINING CONFIGURATION
# ============================================================================
# Configuration: Backend selection and reproducibility
USE_CUCIM = False  # Set to True if CuCIM is available and desired
GLOBAL_SEED = 42

# Model/backbone and precision configs
USE_EFFICIENTNET_B0 = False  # Set True to use EfficientNet-B0 backbone
AMP_ENABLED = True            # Automatic Mixed Precision for faster training
NUM_CLASSES = 2              # Binary classification (HER2-positive vs HER2-negative)
BATCH_SIZE = 16
LR_PHASE1 = 1e-4
LR_PHASE2 = 1e-5
EPOCHS_PHASE1 = 15
EPOCHS_PHASE2 = 30
EARLY_STOP_PATIENCE = 5

In [5]:
# Load dataset from patch extraction CSV
# Use BOTH raw AND normalized patches combined for training

# Available metadata files and Zarr archives (from Section 4.2)
metadata_csv_norm = OUTPUT_BASE / "patch_metadata_512_norm.csv"
metadata_csv_raw = OUTPUT_BASE / "patch_metadata_512_raw.csv"
zarr_path_norm = str(PATCH_ZARR_OUTPUT_NORM)
zarr_path_raw = str(PATCH_ZARR_OUTPUT_RAW)

# Check which files are available
print(f"✓ Checking available metadata files:")
print(f"  Normalized CSV: {metadata_csv_norm.exists()} ({metadata_csv_norm})")
print(f"  Raw CSV: {metadata_csv_raw.exists()} ({metadata_csv_raw})")

# Load normalized patches
if not metadata_csv_norm.exists():
    print(f"\n⚠️  Normalized metadata CSV not found at {metadata_csv_norm}")
    raise FileNotFoundError(f"Missing metadata: {metadata_csv_norm}")

patch_metadata_norm = pd.read_csv(metadata_csv_norm)
print(f"\n✓ Loaded NORMALIZED patch metadata")
print(f"  File: {metadata_csv_norm}")
print(f"  Total patches: {len(patch_metadata_norm):,}")

# Load raw patches
if not metadata_csv_raw.exists():
    print(f"\n⚠️  Raw metadata CSV not found at {metadata_csv_raw}")
    raise FileNotFoundError(f"Missing metadata: {metadata_csv_raw}")

patch_metadata_raw = pd.read_csv(metadata_csv_raw)
print(f"\n✓ Loaded RAW patch metadata")
print(f"  File: {metadata_csv_raw}")
print(f"  Total patches: {len(patch_metadata_raw):,}")

# Verify Zarr archives exist
try:
    z_norm = zarr.open_array(zarr_path_norm, mode='r')
    print(f"\n✓ Normalized Zarr loaded: {zarr_path_norm}")
    print(f"  Shape: {z_norm.shape}")
except Exception as e:
    print(f"\n⚠️  Could not open normalized Zarr: {e}")
    raise

try:
    z_raw = zarr.open_array(zarr_path_raw, mode='r')
    print(f"\n✓ Raw Zarr loaded: {zarr_path_raw}")
    print(f"  Shape: {z_raw.shape}")
except Exception as e:
    print(f"\n⚠️  Could not open raw Zarr: {e}")
    raise

# Combine both datasets
# Add a 'source' column to track origin (for analysis if needed)
patch_metadata_norm['source'] = 'normalized'
patch_metadata_raw['source'] = 'raw'

patch_metadata = pd.concat([patch_metadata_norm, patch_metadata_raw], ignore_index=True)
print(f"\n✓ Combined dataset prepared from BOTH normalized and raw patches")
print(f"  Total patches: {len(patch_metadata):,}")
print(f"  Normalized: {len(patch_metadata_norm):,} ({100*len(patch_metadata_norm)/len(patch_metadata):.1f}%)")
print(f"  Raw: {len(patch_metadata_raw):,} ({100*len(patch_metadata_raw)/len(patch_metadata):.1f}%)")
print(f"  Columns: {list(patch_metadata.columns)}")

# NOTE: Training will use BOTH Zarr archives
# The indices will be remapped during training to handle both datasets
# For now, use normalized Zarr and note that raw patches are available separately
zarr_path = zarr_path_norm  # Will reference normalized as primary
zarr_path_secondary = zarr_path_raw  # Secondary Zarr with raw patches

# Create train/val/test split (70/10/20) based on combined metadata
np.random.seed(SPLIT_SEED)
indices = np.arange(len(patch_metadata))
np.random.shuffle(indices)

# Calculate split indices
train_idx = int(TRAIN_RATIO * len(patch_metadata))
val_idx = int((TRAIN_RATIO + VAL_RATIO) * len(patch_metadata))

train_indices = indices[:train_idx]
val_indices = indices[train_idx:val_idx]
test_indices = indices[val_idx:]

# Display split info
print(f"\n✓ Dataset split (70/10/20) from COMBINED patches:")
print(f"  Train indices: {len(train_indices):,} ({TRAIN_RATIO*100:.0f}%)")
print(f"  Val indices: {len(val_indices):,} ({VAL_RATIO*100:.0f}%)")
print(f"  Test indices: {len(test_indices):,} ({TEST_RATIO*100:.0f}%)")

# Display HER2 distribution
her2_pos = (patch_metadata['her2_status'] == 1).sum()
her2_neg = (patch_metadata['her2_status'] == 0).sum()
print(f"\n  HER2 Status Distribution (Combined):")
print(f"    Positive: {her2_pos:,} patches ({100*her2_pos/len(patch_metadata):.1f}%)")
print(f"    Negative: {her2_neg:,} patches ({100*her2_neg/len(patch_metadata):.1f}%)")

# Distribution by source
print(f"\n  Distribution by source:")
norm_pos = ((patch_metadata['source'] == 'normalized') & (patch_metadata['her2_status'] == 1)).sum()
norm_neg = ((patch_metadata['source'] == 'normalized') & (patch_metadata['her2_status'] == 0)).sum()
raw_pos = ((patch_metadata['source'] == 'raw') & (patch_metadata['her2_status'] == 1)).sum()
raw_neg = ((patch_metadata['source'] == 'raw') & (patch_metadata['her2_status'] == 0)).sum()
print(f"    Normalized+: {norm_pos:,} | Normalized-: {norm_neg:,}")
print(f"    Raw+: {raw_pos:,} | Raw-: {raw_neg:,}")

✓ Checking available metadata files:
  Normalized CSV: True (outputs\preprocessing\patch_metadata_512_norm.csv)
  Raw CSV: True (outputs\preprocessing\patch_metadata_512_raw.csv)

✓ Loaded NORMALIZED patch metadata
  File: outputs\preprocessing\patch_metadata_512_norm.csv
  Total patches: 230,120

✓ Loaded RAW patch metadata
  File: outputs\preprocessing\patch_metadata_512_raw.csv
  Total patches: 914,698

✓ Normalized Zarr loaded: D:\zarr\patches_norm.zarr
  Shape: (230120, 512, 512, 3)

✓ Raw Zarr loaded: D:\zarr\patches_raw.zarr
  Shape: (914698, 512, 512, 3)

✓ Combined dataset prepared from BOTH normalized and raw patches
  Total patches: 1,144,818
  Normalized: 230,120 (20.1%)
  Raw: 914,698 (79.9%)
  Columns: ['slide_name', 'case_name', 'her2_status', 'patch_global_index', 'patch_in_slide', 'x', 'y', 'source']

✓ Dataset split (70/10/20) from COMBINED patches:
  Train indices: 801,372 (70%)
  Val indices: 114,482 (10%)
  Test indices: 228,964 (20%)

  HER2 Status Distribution (C

In [18]:
# ============================================================================
# EXTRACT SAME COORDINATES FROM WSI FILES
# ============================================================================

# Check metadata columns to understand coordinate format
print(f"✓ Test patch metadata columns:")
print(f"  {list(test_patch_metadata_df.columns)}\n")

# Display first few rows to understand the structure
print(f"✓ Sample metadata (first 3 rows):")
print(test_patch_metadata_df.head(3))

# Extract patches from original WSI files using the same coordinates
print(f"\n✓ Extracting patches from WSI files using saved coordinates...")

wsi_output_dir = test_output_dir / "wsi_extracted"
wsi_output_dir.mkdir(parents=True, exist_ok=True)

# Get list of available WSI files
wsi_paths_dict = {}
for cohort_name in ["TCGA_BRCA_Filtered", "Yale_HER2_cohort", "Yale_trastuzumab_response_cohort"]:
    cohort_dir = Path("data") / cohort_name / "SVS"
    if cohort_dir.exists():
        wsi_files = list(cohort_dir.glob("*.svs"))
        for wsi_file in wsi_files:
            slide_name = wsi_file.stem
            wsi_paths_dict[slide_name] = str(wsi_file)
        print(f"  Found {len(wsi_files)} WSI files in {cohort_name}")

print(f"\n✓ Total WSI files available: {len(wsi_paths_dict)}")

# Debug: Show available slide names
if wsi_paths_dict:
    print(f"\n✓ Available slide names (first 10):")
    for i, slide_name in enumerate(list(wsi_paths_dict.keys())[:10]):
        print(f"    {slide_name}")

# Debug: Show unique slide names in metadata
unique_slides_in_meta = test_patch_metadata_df['slide_name'].unique()
print(f"\n✓ Unique slide names in test metadata (first 10):")
for i, slide_name in enumerate(list(unique_slides_in_meta)[:10]):
    print(f"    {slide_name}")

# Extract ALL test patches from WSI
sample_size = len(test_patch_metadata_df)  # Extract all test patches
print(f"\n✓ Extracting {sample_size} test patches from WSI files...")

extracted_count = 0
failed_count = 0

for idx in range(sample_size):
    try:
        patch_row = test_patch_metadata_df.iloc[idx]
        slide_name = patch_row.get('slide_name', None)
        
        # Get patch coordinates - check for common column names
        coord_cols = [col for col in test_patch_metadata_df.columns if 'x' in col.lower() or 'y' in col.lower()]
        print(f"\n  Patch {idx}: Slide Name = {slide_name}")
        print(f"    Coordinate columns: {coord_cols}")
        
        # Try exact match first, then fuzzy match if needed
        if slide_name and slide_name in wsi_paths_dict:
            wsi_path = wsi_paths_dict[slide_name]
        elif slide_name:
            # Try fuzzy matching - look for partial matches
            matching_slides = [s for s in wsi_paths_dict.keys() if str(slide_name).lower() in s.lower() or s.lower() in str(slide_name).lower()]
            if matching_slides:
                wsi_path = wsi_paths_dict[matching_slides[0]]
                print(f"    (Fuzzy matched to: {matching_slides[0]})")
            else:
                print(f"    ⚠️  No matching slide found. Available slides:")
                print(f"       {list(wsi_paths_dict.keys())}")
                failed_count += 1
                continue
        else:
            print(f"    ⚠️  Slide name is None")
            failed_count += 1
            continue
        
        print(f"    WSI path: {wsi_path}")
        
        # Try to open the WSI file
        try:
            reader = WSIReader(wsi_path)
            print(f"    WSI shape: {reader.dimensions}")
            
            # Get coordinates if available
            if len(coord_cols) >= 2:
                x = int(patch_row[coord_cols[0]])
                y = int(patch_row[coord_cols[1]])
                print(f"    Coordinates: ({x}, {y})")
                
                # Extract patch at this coordinate
                patch = reader.read_region(x, y, LEVEL_ANALYSIS, (PATCH_SIZE_TRAIN, PATCH_SIZE_TRAIN))
                print(f"    Extracted patch shape: {patch.shape}")
                print(f"    Patch dtype: {patch.dtype}, range: [{patch.min()}, {patch.max()}]")
                
                # Save extracted patch
                wsi_png_path = wsi_output_dir / f"wsi_patch_idx{idx}_slide{slide_name}.png"
                
                # Handle different data types - ensure we have uint8 in [0, 255] range
                if patch.dtype == np.uint8:
                    patch_uint8 = patch
                elif patch.max() <= 1.0:
                    # Normalize from [0, 1] to [0, 255]
                    patch_uint8 = np.uint8(patch * 255)
                else:
                    # Clip to [0, 255] for other ranges
                    patch_uint8 = np.uint8(np.clip(patch, 0, 255))
                
                mpimg.imsave(str(wsi_png_path), patch_uint8)
                extracted_count += 1
                print(f"    ✓ Saved to: {wsi_png_path.name}")
            else:
                print(f"    ⚠️  Could not find x/y coordinates in metadata")
                failed_count += 1
                
        except Exception as e:
            print(f"    ⚠️  Error reading WSI: {e}")
            failed_count += 1
            
    except Exception as e:
        print(f"  ⚠️  Error processing patch {idx}: {e}")
        failed_count += 1

print(f"\n✓ Extraction complete:")
print(f"  Successfully extracted: {extracted_count}")
print(f"  Failed: {failed_count}")
print(f"  Output directory: {wsi_output_dir}")

✓ Test patch metadata columns:
  ['slide_name', 'case_name', 'her2_status', 'patch_global_index', 'patch_in_slide', 'x', 'y', 'source']

✓ Sample metadata (first 3 rows):
                                          slide_name                case_name  \
0  TCGA-BH-A0C7-01Z-00-DX1.C70D358E-C48F-4F69-86C...  TCGA-BH-A0C7-01Z-00-DX1   
1  TCGA-A2-A04X-01Z-00-DX1.E01A4522-67B3-4FEF-BD6...  TCGA-A2-A04X-01Z-00-DX1   
2  TCGA-BH-A0DK-01Z-00-DX1.0CFED53C-BAD9-4E35-B12...  TCGA-BH-A0DK-01Z-00-DX1   

   her2_status  patch_global_index  patch_in_slide      x      y      source  
0            1              664488           11421  92160  79360         raw  
1            1              352470            2280  40960  58368         raw  
2            0              100482            3637  41472  44544  normalized  

✓ Extracting patches from WSI files using saved coordinates...
  Found 194 WSI files in TCGA_BRCA_Filtered
  Found 192 WSI files in Yale_HER2_cohort
  Found 85 WSI files in Yale_trastuzum

KeyboardInterrupt: 